# ML_Experiments: анализ attention в FLUX и адаптация StyleAligned

Ноутбук продолжает идеи из предзащиты:
- анализ слоев/голов внимания по таймстепам в FLUX;
- поиск `vital` голов по энтропии и стабильности;
- downstream-абляция selective shared-attention.


## Гипотезы

1. В FLUX есть устойчивые головы внимания (`low entropy`, `high stability`), важные для структуры/стиля.
2. Вмешательство только в `vital` головы должно быть лучше, чем грубое вмешательство во все головы.
3. Лучшее окно вмешательства по шагам денойзинга находится в середине процесса.


## Запуск в Colab

1. Откройте ячейку `Colab quick setup` и при необходимости включите `MOUNT_DRIVE`.
2. Если датасет лежит не в стандартном месте, задайте `PIE_BENCH_ROOT_OVERRIDE`.
3. Для полного эксперимента оставьте `RUN_HEAVY = True` (нужен GPU).


In [ ]:
# Colab quick setup (edit these flags before running)
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

AUTO_INSTALL_DEPS = IN_COLAB
MOUNT_DRIVE = False
RUN_HEAVY = True if IN_COLAB else False
PIE_BENCH_ROOT_OVERRIDE = ''  # example: /content/drive/MyDrive/PIE-Bench/data

os.environ['RUN_HEAVY'] = '1' if RUN_HEAVY else '0'
if PIE_BENCH_ROOT_OVERRIDE:
    os.environ['PIE_BENCH_ROOT'] = PIE_BENCH_ROOT_OVERRIDE

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

if IN_COLAB and not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
            print('HF_TOKEN loaded from Colab secrets')
    except Exception:
        print('HF_TOKEN was not found in Colab secrets')


def pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(packages)
    subprocess.check_call(cmd)


if AUTO_INSTALL_DEPS:
    pip_install(['torch>=2.1', 'torchvision', 'pillow', 'matplotlib', 'seaborn', 'pandas', 'scikit-learn'])
    pip_install(['diffusers>=0.30', 'transformers', 'accelerate', 'safetensors'])
    pip_install(['git+https://github.com/openai/CLIP.git', 'lpips'])
    print('Dependencies installed')
else:
    print('Auto-install skipped')


In [ ]:
import os
import json
import math
import random
import inspect
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn.functional as F

try:
    from diffusers import FluxPipeline, FluxImg2ImgPipeline
except Exception as e:
    FluxPipeline = None
    FluxImg2ImgPipeline = None
    print('[warn] FLUX import failed:', e)


def env_flag(name: str, default: str = '0') -> bool:
    return os.environ.get(name, default).strip().lower() in {'1', 'true', 'yes', 'y'}


def detect_pie_root() -> Path:
    candidates = []
    if os.environ.get('PIE_BENCH_ROOT'):
        candidates.append(Path(os.environ['PIE_BENCH_ROOT']))

    cwd = Path.cwd()
    candidates.extend([
        cwd / 'data',
        cwd / 'PIE-Bench' / 'data',
        Path('/content/data'),
        Path('/content/PIE-Bench/data'),
        Path('/content/drive/MyDrive/PIE-Bench/data'),
    ])

    seen = set()
    uniq = []
    for c in candidates:
        key = str(c)
        if key not in seen:
            seen.add(key)
            uniq.append(c)

    for c in uniq:
        if (c / 'mapping_file.json').exists() or (c / 'mapping_file_ti2i_benchmark.json').exists():
            return c

    return Path(os.environ.get('PIE_BENCH_ROOT', str(cwd / 'data')))


@dataclass
class CFG:
    pie_root: Path = field(default_factory=detect_pie_root)
    model_id: str = os.environ.get('FLUX_MODEL_ID', 'black-forest-labs/FLUX.1-schnell')
    hf_token: Optional[str] = os.environ.get('HF_TOKEN')

    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype_name: str = 'float16' if torch.cuda.is_available() else 'float32'

    seed: int = 73
    width: int = 768
    height: int = 768
    num_steps: int = 24
    guidance_scale: float = 3.5

    max_samples: int = 12
    max_attn_modules: int = 40
    max_q_tokens: int = 96
    max_k_tokens: int = 256

    run_heavy: bool = env_flag('RUN_HEAVY', '0')


cfg = CFG()
dtype = getattr(torch, cfg.dtype_name)

print(cfg)
print('Resolved PIE root =', cfg.pie_root)
if not cfg.pie_root.exists():
    print('[warn] PIE root does not exist. Set PIE_BENCH_ROOT_OVERRIDE in the setup cell.')


In [ ]:
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)


def make_generator(seed: int):
    g = torch.Generator(device='cpu')
    g.manual_seed(int(seed))
    return g


def show_images(images: List[Image.Image], titles: Optional[List[str]] = None, cols: int = 3):
    if not images:
        print('[info] no images')
        return
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    if rows == 1:
        axes = np.array([axes])
    for idx in range(rows * cols):
        r, c = divmod(idx, cols)
        ax = axes[r, c]
        ax.axis('off')
        if idx < len(images):
            ax.imshow(images[idx])
            if titles and idx < len(titles):
                ax.set_title(titles[idx], fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
cand_ext = ['.jpg', '.jpeg', '.png', '.webp']


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_pie(pie_root: Path):
    p_main = pie_root / 'mapping_file.json'
    p_ti2i = pie_root / 'mapping_file_ti2i_benchmark.json'
    main = load_json(p_main) if p_main.exists() else {}
    ti2i = load_json(p_ti2i) if p_ti2i.exists() else {}
    if not p_main.exists():
        print('[warn] missing', p_main)
    if not p_ti2i.exists():
        print('[warn] missing', p_ti2i)
    return main, ti2i


def resolve_image_path(rel_path: str, pie_root: Path) -> Path:
    p = Path(rel_path)
    full = (pie_root.parent / p) if 'data' in p.parts else (pie_root / p)
    if full.exists():
        return full
    parent = full.parent
    for ext in cand_ext:
        cand = parent / f"{full.stem}{ext}"
        if cand.exists():
            return cand
    imgs = sorted(parent.glob('*')) if parent.exists() else []
    if imgs:
        return imgs[0]
    raise FileNotFoundError(rel_path)


def sample_mapping(mapping: Dict[str, Dict[str, Any]], n: int, seed: int):
    if not mapping:
        return {}
    keys = sorted(mapping.keys())
    rnd = random.Random(seed)
    rnd.shuffle(keys)
    return {k: mapping[k] for k in keys[:min(n, len(keys))]}


mapping_main, mapping_ti2i = load_pie(cfg.pie_root)
pie_subset = sample_mapping(mapping_main, cfg.max_samples, cfg.seed)

print('PIE main:', len(mapping_main), '| subset:', len(pie_subset), '| ti2i:', len(mapping_ti2i))

preview = []
for sid, item in list(pie_subset.items())[:8]:
    preview.append({
        'sample_id': sid,
        'editing_type_id': item.get('editing_type_id'),
        'original_prompt': item.get('original_prompt', '')[:80],
        'editing_prompt': item.get('editing_prompt', '')[:80],
    })

pd.DataFrame(preview)


In [ ]:
def load_flux_pipes(cfg: CFG):
    if FluxPipeline is None:
        raise RuntimeError('diffusers FLUX classes are unavailable')
    kwargs = {'torch_dtype': dtype}
    if cfg.hf_token:
        kwargs['token'] = cfg.hf_token

    t2i = FluxPipeline.from_pretrained(cfg.model_id, **kwargs).to(cfg.device)

    i2i = None
    if FluxImg2ImgPipeline is not None:
        try:
            i2i = FluxImg2ImgPipeline(**t2i.components).to(cfg.device)
        except Exception as e:
            print('[warn] FluxImg2Img unavailable:', e)
    return t2i, i2i


def attach_callback_if_supported(pipe, kwargs, callback_fn):
    sig = inspect.signature(pipe.__call__)
    if 'callback_on_step_end' in sig.parameters:
        kwargs['callback_on_step_end'] = callback_fn
    if 'callback_on_step_end_tensor_inputs' in sig.parameters:
        kwargs['callback_on_step_end_tensor_inputs'] = ['latents']
    return kwargs


def run_single(pipe_t2i, prompt: str, seed: int, pipe_i2i=None, src_image: Optional[Image.Image]=None, callback_fn=None):
    g = make_generator(seed)
    if pipe_i2i is not None and src_image is not None:
        kwargs = dict(prompt=prompt, image=src_image, strength=0.8, num_inference_steps=cfg.num_steps, guidance_scale=cfg.guidance_scale, generator=g)
        if callback_fn is not None:
            kwargs = attach_callback_if_supported(pipe_i2i, kwargs, callback_fn)
        return pipe_i2i(**kwargs).images[0]

    kwargs = dict(prompt=prompt, width=cfg.width, height=cfg.height, num_inference_steps=cfg.num_steps, guidance_scale=cfg.guidance_scale, generator=g)
    if callback_fn is not None:
        kwargs = attach_callback_if_supported(pipe_t2i, kwargs, callback_fn)
    return pipe_t2i(**kwargs).images[0]


pipe_t2i = None
pipe_i2i = None
if cfg.run_heavy:
    pipe_t2i, pipe_i2i = load_flux_pipes(cfg)
    print('FLUX loaded')
else:
    print('RUN_HEAVY=0 -> heavy cells are skipped')


In [ ]:
def get_transformer(pipe):
    for attr in ('transformer', 'unet', 'model'):
        if hasattr(pipe, attr):
            return getattr(pipe, attr)
    raise AttributeError('No transformer-like module found')


def find_attn_modules(transformer, max_modules: int = 40):
    out = {}
    for name, module in transformer.named_modules():
        if all(hasattr(module, x) for x in ('to_q', 'to_k', 'to_v', 'heads')):
            h = getattr(module, 'heads', None)
            if isinstance(h, int) and h > 0:
                out[name] = module
                if len(out) >= max_modules:
                    break
    return out


class AttentionProbe:
    # collects per-head entropy and temporal stability
    def __init__(self, modules: Dict[str, torch.nn.Module], max_q_tokens: int = 96, max_k_tokens: int = 256):
        self.modules = modules
        self.max_q_tokens = max_q_tokens
        self.max_k_tokens = max_k_tokens
        self.step = 0
        self.records = []
        self.prev = {}
        self.handles = []

    @staticmethod
    def reshape_heads(x: torch.Tensor, heads: int):
        if x.ndim != 3:
            return None
        b, t, c = x.shape
        if c % heads != 0:
            return None
        d = c // heads
        return x.view(b, t, heads, d).permute(0, 2, 1, 3).contiguous()

    @torch.no_grad()
    def capture(self, layer: str, attn, hidden_states: torch.Tensor, encoder_hidden_states: Optional[torch.Tensor]):
        if hidden_states is None or hidden_states.ndim != 3:
            return
        kv = encoder_hidden_states if encoder_hidden_states is not None else hidden_states
        if kv.ndim != 3:
            return

        q = attn.to_q(hidden_states)[:, : self.max_q_tokens, :]
        k = attn.to_k(kv)[:, : self.max_k_tokens, :]

        heads = int(attn.heads)
        qh = self.reshape_heads(q, heads)
        kh = self.reshape_heads(k, heads)
        if qh is None or kh is None:
            return

        logits = torch.einsum('bhqd,bhkd->bhqk', qh.float(), kh.float()) / math.sqrt(qh.shape[-1])
        probs = logits.softmax(dim=-1)

        entropy = -(probs * probs.clamp_min(1e-8).log()).sum(dim=-1)
        entropy = entropy / math.log(max(probs.shape[-1], 2))
        ent_h = entropy.mean(dim=(0, 2)).detach().cpu()

        sig = probs.mean(dim=(0, 2)).detach().cpu()
        if layer in self.prev and self.prev[layer].shape == sig.shape:
            st_h = F.cosine_similarity(sig, self.prev[layer], dim=-1)
        else:
            st_h = torch.full((sig.shape[0],), float('nan'))
        self.prev[layer] = sig

        for h in range(sig.shape[0]):
            self.records.append({'step': int(self.step), 'layer': layer, 'head': int(h), 'entropy': float(ent_h[h]), 'stability': float(st_h[h])})

    def _hook(self, layer: str, attn):
        def fn(module, args, kwargs):
            hidden = args[0] if len(args) > 0 else kwargs.get('hidden_states')
            enc = kwargs.get('encoder_hidden_states')
            try:
                self.capture(layer, attn, hidden, enc)
            except Exception:
                pass
        return fn

    def register(self):
        self.remove()
        self.records = []
        self.prev = {}
        self.step = 0
        for layer, attn in self.modules.items():
            self.handles.append(attn.register_forward_pre_hook(self._hook(layer, attn), with_kwargs=True))

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def callback_on_step_end(self, pipe, step_index: int, timestep, callback_kwargs):
        self.step = int(step_index) + 1
        return callback_kwargs

    def to_df(self):
        cols = ['step', 'layer', 'head', 'entropy', 'stability']
        return pd.DataFrame(self.records, columns=cols)


attn_modules = {}
attn_df = pd.DataFrame(columns=['step', 'layer', 'head', 'entropy', 'stability'])
preview_img = None

if cfg.run_heavy and pipe_t2i is not None:
    transformer = get_transformer(pipe_t2i)
    attn_modules = find_attn_modules(transformer, max_modules=cfg.max_attn_modules)
    print('attention modules:', len(attn_modules))

    probe = AttentionProbe(attn_modules, max_q_tokens=cfg.max_q_tokens, max_k_tokens=cfg.max_k_tokens)

    prompt = 'a cinematic portrait of a fox, watercolor style'
    src = None
    if pie_subset:
        sid, sample = next(iter(pie_subset.items()))
        prompt = sample.get('editing_prompt') or sample.get('original_prompt') or prompt
        img_path = sample.get('image_path')
        if img_path:
            try:
                src_path = resolve_image_path(img_path, cfg.pie_root)
                src = Image.open(src_path).convert('RGB')
                if pipe_i2i is None:
                    cfg.width, cfg.height = src.width, src.height
            except Exception as e:
                print('[warn] source image unavailable:', e)

    probe.register()
    try:
        preview_img = run_single(pipe_t2i, prompt, cfg.seed, pipe_i2i=pipe_i2i, src_image=src, callback_fn=probe.callback_on_step_end)
    finally:
        probe.remove()

    attn_df = probe.to_df()
    print('records:', len(attn_df))
    display(attn_df.head(10))
    if preview_img is not None:
        show_images([preview_img], ['probe generation'], cols=1)
else:
    print('[info] attention probing skipped')


In [ ]:
def summarize_heads(attn_df: pd.DataFrame):
    if attn_df.empty:
        return pd.DataFrame(columns=['layer', 'head', 'mean_entropy', 'std_entropy', 'mean_stability', 'calls'])
    out = (
        attn_df.groupby(['layer', 'head'], as_index=False)
        .agg(mean_entropy=('entropy', 'mean'), std_entropy=('entropy', 'std'), mean_stability=('stability', 'mean'), calls=('entropy', 'size'))
        .fillna({'std_entropy': 0.0})
    )
    return out


head_stats = summarize_heads(attn_df)
print('head stats:', len(head_stats))
display(head_stats.head(10))

if not attn_df.empty:
    top_layers = attn_df.groupby('layer')['stability'].mean().sort_values(ascending=False).head(6).index.tolist()
    plot_df = attn_df[attn_df['layer'].isin(top_layers)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    p1 = plot_df.groupby(['step', 'layer'], as_index=False)['entropy'].mean()
    p2 = plot_df.groupby(['step', 'layer'], as_index=False)['stability'].mean()

    sns.lineplot(data=p1, x='step', y='entropy', hue='layer', ax=axes[0])
    axes[0].set_title('Entropy over timesteps')
    sns.lineplot(data=p2, x='step', y='stability', hue='layer', ax=axes[1])
    axes[1].set_title('Stability over timesteps')

    for ax in axes:
        ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


def cluster_and_select_vital(head_stats: pd.DataFrame, n_clusters: int = 3, st_q: float = 0.75, ent_q: float = 0.35):
    if head_stats.empty:
        return head_stats.copy(), pd.DataFrame(), {}

    clustered = head_stats.copy()
    X = clustered[['mean_entropy', 'std_entropy', 'mean_stability']].fillna(0.0).to_numpy()
    X = StandardScaler().fit_transform(X)
    clustered['cluster'] = KMeans(n_clusters=n_clusters, random_state=cfg.seed, n_init=20).fit_predict(X)

    st_thr = clustered['mean_stability'].quantile(st_q)
    ent_thr = clustered['mean_entropy'].quantile(ent_q)

    vital = clustered[(clustered['mean_stability'] >= st_thr) & (clustered['mean_entropy'] <= ent_thr)].copy()
    vital_map = vital.sort_values(['layer', 'head']).groupby('layer')['head'].apply(list).to_dict()
    return clustered, vital, vital_map


head_clustered, vital_df, vital_map = cluster_and_select_vital(head_stats)
print('clustered:', len(head_clustered), '| vital:', len(vital_df), '| layers with vital:', len(vital_map))
if not head_clustered.empty:
    display(head_clustered.sort_values(['cluster', 'mean_stability'], ascending=[True, False]).head(20))
if not vital_df.empty:
    display(vital_df.sort_values(['mean_stability', 'mean_entropy'], ascending=[False, True]).head(20))


In [ ]:
class SharedKVController:
    # shares K/V for selected heads from style anchor (batch item 0) to other items
    def __init__(self, vital_map: Dict[str, List[int]], total_steps: int, step_window: Tuple[float, float] = (0.15, 0.75), share_values: bool = True):
        self.vital_map = vital_map
        self.total_steps = max(1, int(total_steps))
        self.step_window = step_window
        self.share_values = share_values
        self.step = 0
        self.handles = []

    def active(self):
        if self.total_steps <= 1:
            return True
        frac = self.step / float(self.total_steps - 1)
        return self.step_window[0] <= frac <= self.step_window[1]

    def _make_hook(self, layer: str, heads: int):
        chosen = sorted({h for h in self.vital_map.get(layer, []) if 0 <= h < heads})
        if not chosen:
            return None
        idx_cpu = torch.tensor(chosen, dtype=torch.long)

        def hook(module, args, output):
            if not isinstance(output, torch.Tensor) or output.ndim != 3 or output.shape[0] < 2 or not self.active():
                return output
            b, t, c = output.shape
            if c % heads != 0:
                return output
            d = c // heads
            x = output.view(b, t, heads, d)
            idx = idx_cpu.to(x.device)
            src = x[0:1, :, idx, :]
            x[1:, :, idx, :] = src.expand(b - 1, -1, -1, -1)
            return x.view(b, t, c)
        return hook

    def register(self, attn_modules: Dict[str, torch.nn.Module]):
        self.remove()
        self.step = 0
        for layer, attn in attn_modules.items():
            if layer not in self.vital_map:
                continue
            heads = int(getattr(attn, 'heads', 0))
            if heads <= 0:
                continue
            hk = self._make_hook(layer, heads)
            if hk is not None:
                self.handles.append(attn.to_k.register_forward_hook(hk))
            if self.share_values:
                hv = self._make_hook(layer, heads)
                if hv is not None:
                    self.handles.append(attn.to_v.register_forward_hook(hv))

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def callback_on_step_end(self, pipe, step_index: int, timestep, callback_kwargs):
        self.step = int(step_index) + 1
        return callback_kwargs


def run_batch(pipe, prompts: List[str], seed: int, controller: Optional[SharedKVController] = None):
    kwargs = dict(prompt=prompts, width=cfg.width, height=cfg.height, num_inference_steps=cfg.num_steps, guidance_scale=cfg.guidance_scale, generator=make_generator(seed))
    if controller is not None:
        kwargs = attach_callback_if_supported(pipe, kwargs, controller.callback_on_step_end)
    return pipe(**kwargs).images


style_prompt = 'a still-life setup, watercolor paper texture, soft pastel palette'
content_prompts = [
    'a red vintage car parked near a cafe',
    'a mountain village with a river',
    'a portrait of a corgi in a yellow raincoat',
    'an astronaut riding a bicycle in a city',
]
batch_prompts = [style_prompt] + [f"{p}, same watercolor style" for p in content_prompts]

baseline_images = []
selective_images = []
working_vital = dict(vital_map)

if cfg.run_heavy and pipe_t2i is not None:
    baseline_images = run_batch(pipe_t2i, batch_prompts, cfg.seed, controller=None)

    if not working_vital:
        fallback_layers = list(attn_modules.keys())[:min(4, len(attn_modules))]
        for lname in fallback_layers:
            h = int(getattr(attn_modules[lname], 'heads', 0))
            working_vital[lname] = list(range(min(4, h)))
        print('[info] fallback vital map used')

    ctrl = SharedKVController(working_vital, total_steps=cfg.num_steps, step_window=(0.15, 0.75), share_values=True)
    ctrl.register(attn_modules)
    try:
        selective_images = run_batch(pipe_t2i, batch_prompts, cfg.seed, controller=ctrl)
    finally:
        ctrl.remove()

    show_images(baseline_images, ['baseline'] + [f'b{i}' for i in range(1, len(baseline_images))], cols=3)
    show_images(selective_images, ['selective'] + [f's{i}' for i in range(1, len(selective_images))], cols=3)
else:
    print('[info] style ablation skipped')


In [ ]:
try:
    import clip
except Exception as e:
    clip = None
    print('[warn] clip import failed:', e)

try:
    import lpips
except Exception as e:
    lpips = None
    print('[warn] lpips import failed:', e)

import torchvision.transforms as T


def init_metrics(device: str):
    out = {}
    if clip is not None:
        m, pp = clip.load('ViT-B/32', device=device)
        out['clip_model'] = m
        out['clip_preprocess'] = pp
    if lpips is not None:
        out['lpips'] = lpips.LPIPS(net='alex').to(device)
    return out


def clip_text_sim(bundle, image: Image.Image, text: str, device: str):
    if 'clip_model' not in bundle:
        return float('nan')
    m = bundle['clip_model']
    pp = bundle['clip_preprocess']
    with torch.no_grad():
        it = pp(image).unsqueeze(0).to(device)
        tt = clip.tokenize([text]).to(device)
        fi = m.encode_image(it)
        ft = m.encode_text(tt)
        fi = fi / fi.norm(dim=-1, keepdim=True)
        ft = ft / ft.norm(dim=-1, keepdim=True)
        return float((fi * ft).sum(dim=-1).item())


def clip_img_sim(bundle, img1: Image.Image, img2: Image.Image, device: str):
    if 'clip_model' not in bundle:
        return float('nan')
    m = bundle['clip_model']
    pp = bundle['clip_preprocess']
    with torch.no_grad():
        t1 = pp(img1).unsqueeze(0).to(device)
        t2 = pp(img2).unsqueeze(0).to(device)
        f1 = m.encode_image(t1)
        f2 = m.encode_image(t2)
        f1 = f1 / f1.norm(dim=-1, keepdim=True)
        f2 = f2 / f2.norm(dim=-1, keepdim=True)
        return float((f1 * f2).sum(dim=-1).item())


def lpips_dist(bundle, img1: Image.Image, img2: Image.Image, device: str):
    if 'lpips' not in bundle:
        return float('nan')
    tr = T.Compose([T.Resize((256, 256), interpolation=T.InterpolationMode.BICUBIC), T.ToTensor()])
    t1 = (tr(img1.convert('RGB')).unsqueeze(0).to(device) * 2 - 1)
    t2 = (tr(img2.convert('RGB')).unsqueeze(0).to(device) * 2 - 1)
    with torch.no_grad():
        return float(bundle['lpips'](t1, t2).item())


def evaluate(images, prompts, bundle, device, baseline=None):
    if len(images) <= 1:
        return pd.DataFrame()
    style_ref = images[0]
    rows = []
    for i in range(1, len(images)):
        row = {
            'idx': i,
            'prompt': prompts[i],
            'clip_text': clip_text_sim(bundle, images[i], prompts[i], device),
            'clip_style_ref': clip_img_sim(bundle, images[i], style_ref, device),
        }
        if baseline is not None and i < len(baseline):
            row['lpips_vs_baseline'] = lpips_dist(bundle, images[i], baseline[i], device)
        rows.append(row)
    return pd.DataFrame(rows)


baseline_scores = pd.DataFrame()
selective_scores = pd.DataFrame()
comparison = pd.DataFrame()
summary = pd.DataFrame()

if baseline_images and selective_images:
    metrics = init_metrics(cfg.device)

    baseline_scores = evaluate(baseline_images, batch_prompts, metrics, cfg.device)
    baseline_scores['mode'] = 'baseline'

    selective_scores = evaluate(selective_images, batch_prompts, metrics, cfg.device, baseline=baseline_images)
    selective_scores['mode'] = 'selective_shared_kv'

    comparison = pd.concat([baseline_scores, selective_scores], ignore_index=True)
    metric_cols = [c for c in ['clip_text', 'clip_style_ref', 'lpips_vs_baseline'] if c in comparison.columns]
    summary = comparison.groupby('mode', as_index=False)[metric_cols].mean(numeric_only=True) if metric_cols else pd.DataFrame()

    display(comparison)
    display(summary)
else:
    print('[info] metric evaluation skipped')


In [ ]:
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

saved = []
for name, df in [
    ('flux_attention_raw.csv', attn_df),
    ('flux_head_stats.csv', head_stats),
    ('flux_head_stats_clustered.csv', head_clustered),
    ('flux_vital_heads.csv', vital_df),
    ('style_ablation_per_sample.csv', comparison),
    ('style_ablation_summary.csv', summary),
]:
    if isinstance(df, pd.DataFrame) and not df.empty:
        p = results_dir / name
        df.to_csv(p, index=False)
        saved.append(p)

print('Saved artifacts:')
for p in saved:
    print('-', p)


## Выводы (заполнить после прогона)

1. Какие `layer/head` стабильно попали в vital-набор?
2. Какое окно таймстепов дало лучший компромисс по стилю/содержанию?
3. Есть ли прирост `clip_style_ref` у `selective_shared_kv` относительно `baseline`?
4. Какие ограничения остаются (single-seed, объем выборки, версия модели)?
